# ✈️ AI Travel Planner: API Verification & Interview Walkthrough

This notebook verifies the **4 core external APIs** powering our Agentic Travel Concierge.

| API | Purpose in Architecture | Free Tier Benefit |
| :--- | :--- | :--- |
| **1. Google Gemini** | Core Reasoning LLM & Itinerary Generation | 1,500 requests/day, fast response |
| **2. Tavily Search** | Real-time web discovery for attractions & dining | Built specifically for AI agents |
| **3. OpenWeatherMap** | Live temperature & forecast for packing advice | Global coverage, instant lookup |
| **4. ExchangeRate-API** | Dynamic multi-currency trip budget conversion | Up-to-date daily forex rates |

## 0. Setup: Load Environment Variables

In [7]:
import os
import requests
from dotenv import load_dotenv

# Load secret keys from .env file
load_dotenv()
print("Environment keys loaded successfully!")

Environment keys loaded successfully!


---
## 1. Google Gemini (The Brain / LLM)
**Interview talking point:** *"We use Gemini 3.1 Flash-Lite because it delivers high-speed reasoning (under 1.5s) and has a generous 1,500 free requests per day limit."*

In [8]:
key = os.getenv("GEMINI_API_KEY")
url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent?key={key}"

response = requests.post(url, json={"contents": [{"parts": [{"text": "Say hello in 3 words"}]}]})

print("Status Code:", response.status_code)
print("Gemini Output:", response.json()["candidates"][0]["content"]["parts"][0]["text"].strip())

Status Code: 200
Gemini Output: Hello, how are?


---
## 2. Tavily AI Search (The Eyes / Web Research)
**Interview talking point:** *"Instead of basic Google search, Tavily is an AI search engine built specifically for autonomous agents to return clean, factual snippets without ads or scraping noise."*

In [9]:
response = requests.post(
    "https://api.tavily.com/search",
    json={
        "api_key": os.getenv("TAVILY_API_KEY"),
        "query": "Top attractive places in Goa",
        "max_results": 2
    }
)

print("Status Code:", response.status_code)
for place in response.json().get("results", []):
    print(f"- {place['title']}: {place['url']}")

Status Code: 200
- Top 10 Famous Places to visit in Goa: https://www.rvcj.com/top-10-famous-places-visit-goa
- Top 10 Must-Visit Places in Goa for an Unforgettable Vacation: https://myholidayhappiness.com/places-to-visit/goa/goa-tourism


---
## 3. OpenWeatherMap (The Advisor / Weather & Packing)
**Interview talking point:** *"The agent checks real-time weather to provide practical, destination-specific advice like packing breathable cottons for Goa or thermal jackets for winter trips."*

In [10]:
response = requests.get(
    "https://api.openweathermap.org/data/2.5/weather",
    params={
        "q": "Goa",
        "appid": os.getenv("OPENWEATHERMAP_API_KEY"),
        "units": "metric"
    }
)

print("Status Code:", response.status_code)
weather_data = response.json()
print(f"City: {weather_data['name']}")
print(f"Temperature: {weather_data['main']['temp']}°C")
print(f"Condition: {weather_data['weather'][0]['description']}")

Status Code: 200
City: Goa
Temperature: 31.04°C
Condition: overcast clouds


---
## 4. ExchangeRate-API (The Budget Calculator)
**Interview talking point:** *"Travelers often think in their home currency. This tool dynamically converts local costs (e.g. INR in Goa) back to USD or EUR for realistic budget planning."*

In [11]:
key = os.getenv("EXCHANGE_RATE_API_KEY")
response = requests.get(f"https://v6.exchangerate-api.com/v6/{key}/pair/USD/INR/100")

print("Status Code:", response.status_code)
rate_data = response.json()
print(f"100 USD = ₹{rate_data.get('conversion_result')} INR (Exchange Rate: {rate_data.get('conversion_rate')})")

Status Code: 200
100 USD = ₹9591.87 INR (Exchange Rate: 95.9187)


In [12]:
import pytest
from utils.expense_calculator import Calculator

# @pytest.mark.parametrize runs the same test function multiple times
# with different sets of inputs (total, days, expected).
@pytest.mark.parametrize("total, days, expected", [
    (100.0, 5, 20.0),   # Case 1: Normal calculation
    (100.0, 0, 0.0),    # Case 2: Division by zero edge case
    (0.0, 10, 0.0),     # Case 3: Zero total
])
def test_daily_budget(total, days, expected):
    result = Calculator.calculate_daily_budget(total, days)
    assert result == expected
